# NYC Borough and Subway Station GeoJSON Collection

## Imports and Folder Setup

In [26]:
import requests
import json
import os
import zipfile
import io
import geopandas as gpd
import pandas as pd

notebooks_folder = os.getcwd() 
raw_root_folder = os.path.abspath( os.path.join(notebooks_folder, "..", "data", "raw"))
interim_root_folder = os.path.abspath( os.path.join(notebooks_folder, "..", "data", "interim"))
processed_root_folder = os.path.abspath( os.path.join(notebooks_folder, "..", "data", "processed"))

## Download NYC Borough GeoJSON

In [27]:
# Download NYC borough boundaries # Clipped to Shoreline 
borough_url = "https://services5.arcgis.com/GfwWNkhOj9bNBqoJ/arcgis/rest/services/NYC_Borough_Boundary/FeatureServer/0/query?where=1=1&outFields=*&outSR=4326&f=geojson"

response = requests.get(borough_url)
borough_geojson = response.json()

# borough_path = os.path.join(raw_root_folder, "nyc_boroughs.geojson")
borough_path = os.path.join(processed_root_folder, "New York City Boroughs.geojson")
with open(borough_path, 'w') as f:
    json.dump(borough_geojson, f)

print(f"Downloaded: {len(borough_geojson['features'])} boroughs")


Downloaded: 5 boroughs


## Download Subway Stations Data and Create GeoJSON

In [28]:
# Setup paths
subway_raw_folder = os.path.join(raw_root_folder, "New York City Subway Stations")
os.makedirs(subway_raw_folder, exist_ok=True)

# Download MTA GTFS data
gtfs_url = "http://web.mta.info/developers/data/nyct/subway/google_transit.zip"
response = requests.get(gtfs_url)

# Extract stops.txt from zip
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    with z.open('stops.txt') as f:
        stops_df = pd.read_csv(f)

# Save raw data
raw_stops_path = os.path.join(subway_raw_folder, "stops.csv")
stops_df.to_csv(raw_stops_path, index=False)

print(f"Downloaded {len(stops_df)} stop records")
stops_df.head()


nybb_url = "https://s-media.nyc.gov/agencies/dcp/assets/files/zip/data-tools/bytes/borough-boundaries/nybb_25c.zip"
nybb_zip_path =  os.path.join(os.path.join(raw_root_folder, "New York City Subway Stations"), "nybb_25c.zip")
nybb_extract_dir = os.path.join(os.path.join(raw_root_folder, "New York City Subway Stations"), "nybb_25c")

if not os.path.exists(nybb_zip_path):
    print("Downloading borough boundaries shapefile...")
    r = requests.get(nybb_url)
    with open(nybb_zip_path, "wb") as f:
        f.write(r.content)
else:
    print("File already exists:", nybb_zip_path)

# Extract
with zipfile.ZipFile(nybb_zip_path, "r") as z:
    z.extractall(nybb_extract_dir)

# The zip contains a nested folder also named nybb_25c
inner_dir = os.path.join(nybb_extract_dir, "nybb_25c")

print("Inner dir:", inner_dir)
print("Files:", os.listdir(inner_dir))

shp_path = os.path.join(inner_dir, "nybb.shp")

borough_gdf = gpd.read_file(shp_path)
print(borough_gdf.crs)
print(borough_gdf.head())

# Paths
nybb_outer =  os.path.join(os.path.join(raw_root_folder, "New York City Subway Stations"), "nybb_25c")
nybb_inner = os.path.join(nybb_outer, "nybb_25c")
nybb_shp = os.path.join(nybb_inner, "nybb.shp")

# Load borough polygons
borough_gdf = gpd.read_file(nybb_shp)
borough_gdf = borough_gdf.to_crs(epsg=4326)
borough_gdf = borough_gdf[["BoroName", "geometry"]]

# Load subway parent stations
stops_path = os.path.join(subway_raw_folder, "stops.csv")
stops_df = pd.read_csv(stops_path)

stations = stops_df[stops_df["location_type"] == 1].copy()
stations = stations[["stop_id", "stop_name", "stop_lat", "stop_lon"]]

stations_gdf = gpd.GeoDataFrame(
    stations,
    geometry=gpd.points_from_xy(stations["stop_lon"], stations["stop_lat"]),
    crs="EPSG:4326"
)

# Spatial join
joined = gpd.sjoin(
    stations_gdf,
    borough_gdf,
    how="left",
    predicate="within"
)

stations["borough"] = joined["BoroName"].fillna("Unknown")

print("\nStations by borough:")
print(stations["borough"].value_counts(dropna=False))

# Save as GeoJSON
subway_geojson = {
    "type": "FeatureCollection",
    "features": []
}

for _, row in stations.iterrows():
    subway_geojson["features"].append({
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [row["stop_lon"], row["stop_lat"]]
        },
        "properties": {
            "name": row["stop_name"],
            "stop_id": row["stop_id"],
            "borough": row["borough"]
        }
    })

output_path = os.path.join(processed_root_folder, "New York City Subway Stations.geojson")


with open(output_path, "w") as f:
    json.dump(subway_geojson, f)

print("\nSaved GeoJSON:", output_path)

Downloaded 1488 stop records
File already exists: /Users/KailenMcCauley/Documents/Georgia Tech/CSE 6242/NYC-Affordability-Transit-Access/data/raw/New York City Subway Stations/nybb_25c.zip
Inner dir: /Users/KailenMcCauley/Documents/Georgia Tech/CSE 6242/NYC-Affordability-Transit-Access/data/raw/New York City Subway Stations/nybb_25c/nybb_25c
Files: ['nybb.shx', 'nybb.shp', 'nybb.dbf', 'nybb.prj', 'nybb.shp.xml']
EPSG:2263
   BoroCode       BoroName     Shape_Leng    Shape_Area  \
0         5  Staten Island  325912.288988  1.623618e+09   
1         2          Bronx  463147.071960  1.187199e+09   
2         3       Brooklyn  726953.045551  1.934463e+09   
3         4         Queens  887905.078760  3.041419e+09   
4         1      Manhattan  359193.930112  6.366278e+08   

                                            geometry  
0  MULTIPOLYGON (((970217.022 145643.332, 970227....  
1  MULTIPOLYGON (((1012821.806 229228.265, 101278...  
2  MULTIPOLYGON (((1022227.32 152028.146, 1022078...  

In [29]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import os

all_sales = []

nyc_sales_data_path = os.path.join(interim_root_folder, "New York City Sales Data")
for year in os.listdir(nyc_sales_data_path):

    year_folder = os.path.join(nyc_sales_data_path, year)
    if not os.path.isdir(year_folder):
        continue

    for file in os.listdir(year_folder):
        if not file.lower().endswith((".xlsx")) or file.startswith("~"):
            continue

        file_path = os.path.join(year_folder, file)
        df = pd.read_excel(file_path)
        df["SALE_YEAR"] = pd.to_datetime(df["SALE DATE"]).dt.year
        df["BOROUGH_FILE"] = file.removesuffix(".xlsx") 
        all_sales.append(df)

sales_df = pd.concat(all_sales, ignore_index=True)
print("Total sales rows:", len(sales_df))
print(sales_df.head())

# 1) GeoDataFrame from sales_df
houses_gdf = gpd.GeoDataFrame(
    sales_df,
    geometry=gpd.points_from_xy(sales_df["LON"], sales_df["LAT"]),
    crs="EPSG:4326"
)

# Optional: clean up SALE_DATE as datetime (SALE_YEAR already exists)
houses_gdf["SALE_DATE"] = pd.to_datetime(houses_gdf["SALE DATE"], errors="coerce")

# 2) Attach borough polygons
houses_gdf = gpd.sjoin(
    houses_gdf,
    borough_gdf[["BoroName", "geometry"]],
    how="left",
    predicate="within"
).rename(columns={"BoroName": "borough"})

if "index_right" in houses_gdf.columns:
    houses_gdf = houses_gdf.drop(columns=["index_right"])

# 3) Nearest subway station (work in projected CRS – meters)
houses_proj   = houses_gdf.to_crs(epsg=6539)
stations_proj = stations_gdf.to_crs(epsg=6539)

houses_nearest = gpd.sjoin_nearest(
    houses_proj,
    stations_proj[["stop_id", "stop_name", "geometry"]],
    how="left",
    distance_col="dist_m"
)

# Back to WGS84
houses_nearest = houses_nearest.to_crs(epsg=4326)
houses_nearest["dist_km"] = houses_nearest["dist_m"] / 1000.0

# 4) Columns to keep for the map
houses_for_geojson = houses_nearest[[
    "FULL ADDRESS",
    "borough",
    "stop_id",
    "stop_name",
    "dist_m",
    "dist_km",
    "LAT",
    "LON",
    "SALE PRICE",
    "SALE DATE",   # original string/date column
    "SALE_YEAR",
    "geometry"
]].rename(columns={
    "FULL ADDRESS": "address",
    "stop_name": "nearest_station"
})

# 5) Light cleaning before writing (optional but safer)
g = houses_for_geojson.copy()
g = g[g.geometry.notnull()]
g = g[g.geometry.is_valid]
g = g[
    (g.geometry.x.between(-80, -70)) &
    (g.geometry.y.between(35, 45))
]

# ensure types are JSON-friendly
g["SALE DATE"] = pd.to_datetime(g["SALE DATE"], errors="coerce").dt.strftime("%Y-%m-%d")
g["SALE PRICE"] = pd.to_numeric(g["SALE PRICE"], errors="coerce")

houses_geojson_path = os.path.join(processed_root_folder, "New York City Houses.geojson")
g = g.rename(columns={
    "SALE PRICE": "SALE_PRICE",
    "SALE DATE": "SALE_DATE"
})
g.to_file(houses_geojson_path, driver="GeoJSON")
print("Saved houses GeoJSON:", houses_geojson_path)

Total sales rows: 474736
                                        FULL ADDRESS  SALE PRICE  SALE DATE  \
0    65 Avenue D, Alphabet City, New York, NY, 10009     1560000 2013-07-18   
1  243 7Th Street, Alphabet City, New York, NY, 1...     3150000 2013-03-06   
2  238 4Th Street, Alphabet City, New York, NY, 1...     3450000 2013-03-27   
3    17 Avenue B, Alphabet City, New York, NY, 10009         283 2013-04-18   
4    14 Avenue B, Alphabet City, New York, NY, 10009    13185684 2013-01-31   

         LAT        LON  SALE_YEAR BOROUGH_FILE  
0  40.720314 -73.864704       2013    Manhattan  
1  40.672668 -73.990010       2013    Manhattan  
2  40.888990 -73.912779       2013    Manhattan  
3  40.783730 -73.908361       2013    Manhattan  
4  40.786004 -73.850281       2013    Manhattan  
Saved houses GeoJSON: /Users/KailenMcCauley/Documents/Georgia Tech/CSE 6242/NYC-Affordability-Transit-Access/data/processed/New York City Houses.geojson
